# Citi Bike NYC: дистанции поездок

Расчёт расстояния каждой поездки по координатам станций (формула гаверсинусов) средствами Spark SQL, без UDF, и описательная статистика по дистанциям.

In [ ]:
# Зависимости: Java 11+ и PySpark
%pip install -q pyspark==3.5.1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("CitiBikeTripDistance") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import builtins

df = spark.read.option("header", "true").csv("data/201902-citibike-tripdata.csv")

df = df.withColumn("start_lat", F.col("start station latitude").cast(DoubleType())) \
       .withColumn("start_lon", F.col("start station longitude").cast(DoubleType())) \
       .withColumn("end_lat",   F.col("end station latitude").cast(DoubleType())) \
       .withColumn("end_lon",   F.col("end station longitude").cast(DoubleType()))

df = df.dropna(subset=["start_lat", "start_lon", "end_lat", "end_lon"])

# Исключение поездок, где начало и конец - одно и то же
df = df.filter((F.col("start_lat") != F.col("end_lat")) | (F.col("start_lon") != F.col("end_lon")))

R = 6371000.0
PI = 3.141592653589793

def rad(col):
    return col * F.lit(PI / 180.0)

df = df.withColumn("lat1", rad(F.col("start_lat"))) \
       .withColumn("lat2", rad(F.col("end_lat"))) \
       .withColumn("dlat", rad(F.col("end_lat") - F.col("start_lat"))) \
       .withColumn("dlon", rad(F.col("end_lon") - F.col("start_lon")))

# Формула Гаверсина для нахождения кратчайшего расстояния между двумя точками с учетом кривизны земли
a = (
    F.sin(F.col("dlat") / 2) ** 2 +
    F.cos(F.col("lat1")) * F.cos(F.col("lat2")) * F.sin(F.col("dlon") / 2) ** 2
)

a_safe = F.least(F.greatest(a, F.lit(0.0)), F.lit(1.0))

df = df.withColumn("distance_m", 2 * R * F.asin(F.sqrt(a_safe)))
print("Примеры рассчитанных дистанций поездок в метрах:")
df.select("start_lat", "start_lon", "end_lat", "end_lon", "distance_m").show(10, truncate=False)

# Статистика
stats = df.select(
    F.max("distance_m").alias("max"),
    F.mean("distance_m").alias("mean"),
    F.stddev("distance_m").alias("stddev")
).collect()[0]

print("Максимальное значение:", builtins.round(stats["max"], 2), "м")
print("Среднее значение:", builtins.round(stats["mean"], 2), "м")
print("Стандартное отклонение:", builtins.round(stats["stddev"], 2), "м")
print("Медиана:", builtins.round(df.approxQuantile("distance_m", [0.5], 0.01)[0], 2), "м")

Примеры рассчитанных дистанций поездок в метрах:
+-----------------+------------------+-----------------+------------------+------------------+
|start_lat        |start_lon         |end_lat          |end_lon           |distance_m        |
+-----------------+------------------+-----------------+------------------+------------------+
|40.797911        |-73.9423          |40.8014866       |-73.9442507       |430.1604184627014 |
|40.72779126      |-73.98564945      |40.7284186       |-73.98713956      |143.63994294033108|
|40.676368        |-73.952918        |40.678045        |-73.962408        |821.7251348321905 |
|40.7489006       |-73.97604882      |40.75640548      |-73.9900262       |1443.1216853333028|
|40.76309387270797|-73.9783501625061 |40.76132983124814|-73.97982001304626|231.94966788217462|
|40.70924826276157|-73.96063148975372|40.72036775298455|-73.96165072917938|1239.411464511852 |
|40.72228087      |-73.97668709      |40.72453734      |-73.98185424      |502.55733659811364|
|

## Выводы

- Максимальная дистанция **15.3 км**, средняя **1.64 км** (перемещение в пределах района).
- Медиана **1.26 км**, то есть большинство поездок короче 1.3 км.
- Стандартное отклонение **1.29 км**. Разброс большой, распределение с длинным правым хвостом.
- Поездки с одинаковой начальной и конечной станцией исключены, поэтому нулевых дистанций нет.